# **Raw Data Profiling**

## 🎯 Objective

The objective of this notebook is to perform data profiling using PySpark to understand the structure and quality of the dataset.

The profiling includes checking schema, row counts, null values, duplicate keys, distinct values, and min/max values to identify potential data quality issues before further processing.



Spark Session

In [114]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName('raw_data-profiling').getOrCreate()
spark.version

'4.0.4'

Load Raw Datasets

In [115]:
customers = spark.read.csv('/content/raw/customers.csv', header=True, inferSchema=True)
orders = spark.read.csv('/content/raw/orders.csv', header=True, inferSchema=True)
products = spark.read.json('/content/raw/products.json')
transactions = spark.read.csv('/content/raw/transactions.csv', header=True, inferSchema=True)

#### **Customer Profiling**

Customer Schema

In [116]:
customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- account_status: string (nullable = true)



Customer Data Sample

In [117]:
customers.show(5)

+-----------+----------+---------+--------------------+-------------+-------+-----------+----------------+--------------+
|customer_id|first_name|last_name|               email|      country|   city|signup_date|customer_segment|account_status|
+-----------+----------+---------+--------------------+-------------+-------+-----------+----------------+--------------+
|    C000001|       Mia|    Davis|mia.davis1@exampl...|      Germany| Munich| 2024-10-14|  Small Business|        Active|
|    C000002|     Arjun|    Davis|arjun.davis2@exam...|       Canada|Calgary| 2025-08-11|        Consumer|        Active|
|    C000003|      Noah|   Wilson|noah.wilson3@exam...|       Canada|Calgary| 2021-02-24|        Consumer|        Active|
|    C000004|      Liam|   Taylor|liam.taylor4@exam...|United States| Denver| 2022-12-17|        Consumer|      Inactive|
|    C000005|    Sophia|    Smith|sophia.smith5@exa...|       Canada|Calgary| 2021-04-05|        Consumer|      Inactive|
+-----------+----------+

Customer Data Stats

In [118]:
count = customers.count()
num_columns = len(customers.columns)
unique_counts_for_key_column = customers.select(F.count_distinct(F.col('customer_id'))).first()[0]
duplicate_key_groups = (
    customers
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
min_signup_date = customers.select(F.min('signup_date')).first()[0]
max_sign_up_date = customers.select(F.max('signup_date')).first()[0]
duplicate_rows = (
    customers
    .count()
    - customers
        .select("customer_id")
        .distinct()
        .count()
)

print(f"Count of row in customers: {count} \n\
number of columns: {num_columns} \n\
distinct count for key column: {unique_counts_for_key_column} \n\
count of duplicate key: {duplicate_key_groups} \n\
minimum signup date: {min_signup_date} \n\
maximum signup date: {max_sign_up_date} \n\
duplicate rows: {duplicate_rows}")


Count of row in customers: 10000 
number of columns: 9 
distinct count for key column: 9999 
count of duplicate key: 1 
minimum signup date: 2021-01-01 
maximum signup date: 2025-13-40 
duplicate rows: 1


Null count of each column in Customer

In [119]:
customers.select([
    F.count(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c).cast("string")) == "") |
            (F.upper(F.trim(F.col(c).cast("string"))) == "UNKNOWN"),
            1
        )
    ).alias(c)
    for c in customers.columns
]).show()

+-----------+----------+---------+-----+-------+----+-----------+----------------+--------------+
|customer_id|first_name|last_name|email|country|city|signup_date|customer_segment|account_status|
+-----------+----------+---------+-----+-------+----+-----------+----------------+--------------+
|          0|         1|        0|    0|      1|   1|          0|               0|             0|
+-----------+----------+---------+-----+-------+----+-----------+----------------+--------------+



Categorical Value in Customers

In [120]:
customers.select('country').distinct().show()

+--------------+
|       country|
+--------------+
|       Germany|
| United States|
|         India|
|       UNKNOWN|
|        Canada|
|     Australia|
|United Kingdom|
+--------------+



In [121]:
customers.select('customer_segment').distinct().show()

+----------------+
|customer_segment|
+----------------+
|        Consumer|
|         Premier|
|      Enterprise|
|  Small Business|
+----------------+



In [122]:
customers.select('account_status').distinct().show()

+--------------+
|account_status|
+--------------+
|     Suspended|
|        Active|
|       Dormant|
|      Inactive|
+--------------+



#### **Product Profiling**

Product Schema

In [123]:
products.printSchema()

root
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- cost_price: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- unit_price: string (nullable = true)



Product Data Sample

In [124]:
products.show(5)

+------+-----------+----------+------------+---------+----------+-------------------+-----------+----------+
| brand|   category|cost_price|created_date|is_active|product_id|       product_name|subcategory|unit_price|
+------+-----------+----------+------------+---------+----------+-------------------+-----------+----------+
|  Nova|     Beauty|     14.01|  2025-05-01|     true|    P00001|Orion Wellness 0001|   Wellness|     22.83|
| Orion|Electronics|    140.59|  2024-09-06|     true|    P00002| Atlas Laptops 0002|    Laptops|    253.71|
|Harbor|     Beauty|     27.95|  2024-03-07|     true|    P00003|Cedar Wellness 0003|   Wellness|     51.28|
|Harbor|Electronics|    152.65|  2021-10-07|     true|    P00004|  Harbor Audio 0004|      Audio|    180.64|
| Atlas|     Beauty|     27.47|  2021-01-06|     true|    P00005|Cedar Wellness 0005|   Wellness|     43.00|
+------+-----------+----------+------------+---------+----------+-------------------+-----------+----------+
only showing top 5 

Products Data Stats

In [125]:
count = products.count()
num_columns = len(products.columns)
unique_counts_for_key_column = products.select(F.count_distinct(F.col('product_id'))).first()[0]
duplicate_key_count = products.groupBy('product_id').count().filter(F.col("count") > 1).count()
min_created_date = products.select(F.min('created_date')).first()[0]
max_created_date = products.select(F.max('created_date')).first()[0]

print(f"Count of row in products: {count} \n\
number of columns: {num_columns} \n\
distinct count for key column: {unique_counts_for_key_column} \n\
count of duplicate key: {duplicate_key_count} \n\
minimum created date: {min_created_date} \n\
maximum created date: {max_created_date}")

Count of row in products: 1000 
number of columns: 9 
distinct count for key column: 999 
count of duplicate key: 1 
minimum created date: 2020-01-02 
maximum created date: 2025-06-29


Null count of each column in Product

In [126]:
products.select([
    F.count(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c).cast("string")) == "") |
            (F.upper(F.trim(F.col(c).cast("string"))) == "UNKNOWN"),
            1
        )
    ).alias(c)
    for c in products.columns
]).show()

+-----+--------+----------+------------+---------+----------+------------+-----------+----------+
|brand|category|cost_price|created_date|is_active|product_id|product_name|subcategory|unit_price|
+-----+--------+----------+------------+---------+----------+------------+-----------+----------+
|    0|       1|         1|           0|        0|         0|           0|          0|         0|
+-----+--------+----------+------------+---------+----------+------------+-----------+----------+



Categorical Values in Products

In [127]:
products.select('brand').distinct().show()

+------+
| brand|
+------+
|Harbor|
|Vertex|
| Cedar|
|  Luma|
|  Nova|
| Orion|
|  Peak|
| Atlas|
+------+



In [128]:
products.select('category').distinct().show()

+--------------+
|      category|
+--------------+
|Home & Kitchen|
|        Sports|
|   Electronics|
|         Books|
|       UNKNOWN|
|        Beauty|
+--------------+



In [129]:
products.select('subcategory').distinct().show()

+-----------+
|subcategory|
+-----------+
|   Wellness|
|    Outdoor|
|    Fiction|
|    Fitness|
|  Hair Care|
| Sportswear|
|   Lighting|
|  Skin Care|
|   Cookware|
|Accessories|
|  Furniture|
|     Phones|
| Technology|
|   Business|
|    Storage|
|    Cycling|
|      Audio|
|    Laptops|
|   Children|
|     Makeup|
+-----------+



#### **Orders Profiling**

Orders Schema

In [130]:
orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_country: string (nullable = true)
 |-- shipping_city: string (nullable = true)
 |-- order_total: string (nullable = true)



Orders Data Sample

In [131]:
orders.show(5)

+--------+-----------+----------+------------+--------------+----------------+-------------+-----------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_country|shipping_city|order_total|
+--------+-----------+----------+------------+--------------+----------------+-------------+-----------+
|O0000001|    C002471|2024-10-27|   Completed|          Card|       Australia|    Melbourne|    1510.72|
|O0000002|    C009466|2024-04-16|   Completed|           UPI|  United Kingdom|   Manchester|    1716.70|
|O0000003|    C002882|2025-02-19|   Completed|           UPI|          Canada|     Montreal|    1105.60|
|O0000004|    C004694|2025-05-15|   Completed|        Wallet|       Australia|        Perth|    2457.40|
|O0000005|    C009093|2025-01-18|   Completed| Bank Transfer|           India|        Delhi|     480.72|
+--------+-----------+----------+------------+--------------+----------------+-------------+-----------+
only showing top 5 rows


Orders Data Stats

In [132]:
count = orders.count()
num_columns = len(orders.columns)
unique_counts_for_key_column = orders.select(F.count_distinct(F.col('order_id'))).first()[0]
duplicate_key_count = orders.groupBy('order_id').count().filter(F.col("count") > 1).count()
min_order_date = orders.select(F.min('order_date')).first()[0]
max_order_date = orders.select(F.max('order_date')).first()[0]

print(f"Count of row in products: {count} \n\
number of columns: {num_columns} \n\
distinct count for key column: {unique_counts_for_key_column} \n\
count of duplicate key: {duplicate_key_count} \n\
minimum order date: {min_order_date} \n\
maximum order date: {max_order_date}")

Count of row in products: 30000 
number of columns: 8 
distinct count for key column: 29999 
count of duplicate key: 1 
minimum order date: 2024-01-01 
maximum order date: 2025-12-31


Null count of each column in Orders

In [133]:
orders.select([
    F.count(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c).cast("string")) == "") |
            (F.upper(F.trim(F.col(c).cast("string"))) == "UNKNOWN"),
            1
        )
    ).alias(c)
    for c in orders.columns
]).show()

+--------+-----------+----------+------------+--------------+----------------+-------------+-----------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_country|shipping_city|order_total|
+--------+-----------+----------+------------+--------------+----------------+-------------+-----------+
|       0|          0|         0|           0|             0|               1|            1|          0|
+--------+-----------+----------+------------+--------------+----------------+-------------+-----------+



Categorical Value in Orders

In [134]:
orders.select('shipping_country').distinct().show()

+----------------+
|shipping_country|
+----------------+
|         Germany|
|           India|
|   United States|
|         UNKNOWN|
|          Canada|
|       Australia|
|  United Kingdom|
+----------------+



In [135]:
orders.select('shipping_city').distinct().show()

+-------------+
|shipping_city|
+-------------+
|   Manchester|
|      Chennai|
|       Berlin|
|       London|
|       Mumbai|
|     Brisbane|
|     Montreal|
|       Sydney|
|        Perth|
|       Austin|
|      Hamburg|
|      Chicago|
|      Toronto|
|    Vancouver|
|      Seattle|
|      Bristol|
|         Pune|
|        Delhi|
|      Calgary|
|    Melbourne|
+-------------+
only showing top 20 rows


In [136]:
orders.select('order_status').distinct().show()

+------------+
|order_status|
+------------+
|    Returned|
|     Shipped|
|  Processing|
|   Completed|
|   Cancelled|
|  In Transit|
+------------+



In [137]:
orders.select('payment_method').distinct().show()

+--------------+
|payment_method|
+--------------+
| Bank Transfer|
|        Crypto|
|        PayPal|
|          Card|
|        Wallet|
|           UPI|
+--------------+



#### **Transactions Profiling**

Transactions Schema

In [138]:
transactions.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- tax_amount: double (nullable = true)
 |-- transaction_amount: double (nullable = true)
 |-- payment_status: string (nullable = true)



Transaction Data Sample

In [139]:
transactions.show(5)

+--------------+--------+----------+----------------+--------+----------+---------------+----------+------------------+--------------+
|transaction_id|order_id|product_id|transaction_date|quantity|unit_price|discount_amount|tax_amount|transaction_amount|payment_status|
+--------------+--------+----------+----------------+--------+----------+---------------+----------+------------------+--------------+
|     T00000001|O0028299|    P00169|      2024-04-04|       2|     82.22|          16.44|     11.84|            159.84|      Captured|
|     T00000002|O0017225|    P00332|      2024-04-07|       5|     28.76|           7.19|     10.93|            147.54|      Captured|
|     T00000003|O0028084|    P00831|      2025-03-14|       1|    237.10|          35.57|     16.12|            217.66|      Captured|
|     T00000004|O0022477|    P00398|      2024-12-20|       2|    229.73|            0.0|     36.76|            496.22|      Captured|
|     T00000005|O0006141|    P00087|      2024-05-19|  

Transaction Data Stats

In [140]:
count = transactions.count()
num_columns = len(transactions.columns)
unique_counts_for_key_column = transactions.select(F.count_distinct(F.col('transaction_id'))).first()[0]
duplicate_key_count = transactions.groupBy('transaction_id').count().filter(F.col("count") > 1).count()
min_transaction_date = transactions.select(F.min('transaction_date')).first()[0]
max_transaction_date = transactions.select(F.max('transaction_date')).first()[0]

print(f"Count of row in transactions: {count} \n\
number of columns: {num_columns} \n\
distinct count for key column: {unique_counts_for_key_column} \n\
count of duplicate key: {duplicate_key_count} \n\
minimum transactions date: {min_transaction_date} \n\
maximum transactions date: {max_transaction_date}")

Count of row in transactions: 50000 
number of columns: 10 
distinct count for key column: 49999 
count of duplicate key: 1 
minimum transactions date: 2024-01-01 
maximum transactions date: 2025-14-01


Null count of each column in Orders

In [141]:
transactions.select([
    F.count(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c).cast("string")) == "") |
            (F.upper(F.trim(F.col(c).cast("string"))) == "UNKNOWN"),
            1
        )
    ).alias(c)
    for c in transactions.columns
]).show()

+--------------+--------+----------+----------------+--------+----------+---------------+----------+------------------+--------------+
|transaction_id|order_id|product_id|transaction_date|quantity|unit_price|discount_amount|tax_amount|transaction_amount|payment_status|
+--------------+--------+----------+----------------+--------+----------+---------------+----------+------------------+--------------+
|             0|       0|         0|               0|       0|         0|              1|         0|                 0|             0|
+--------------+--------+----------+----------------+--------+----------+---------------+----------+------------------+--------------+



Categorical Value in Transactions

In [142]:
transactions.select('payment_status').distinct().show()

+--------------+
|payment_status|
+--------------+
|       Settled|
|      Refunded|
|        Failed|
|      Captured|
|       Pending|
+--------------+



### Referential-integrity


Orphan Orders

In [143]:
orders.join(
    customers,
    orders.customer_id == customers.customer_id,
    'left_anti'
).count()

5

Orphan Products

In [144]:
transactions.join(
    products,
    transactions.product_id == products.product_id,
    'left_anti'
).count()

49

In [145]:
transactions.join(
    orders,
    transactions.order_id == orders.order_id,
    "left_anti"
).count()

1

Numerical Profiling

In [146]:
products.select(
    F.min("unit_price").alias("min_unit_price"),
    F.max("unit_price").alias("max_unit_price"),
    F.min("cost_price").alias("min_cost_price"),
    F.max("cost_price").alias("max_cost_price")
).show()

+--------------+--------------+--------------+--------------+
|min_unit_price|max_unit_price|min_cost_price|max_cost_price|
+--------------+--------------+--------------+--------------+
|         10.07|           N/A|              |         99.88|
+--------------+--------------+--------------+--------------+



In [147]:
invalid_price_count = products.filter(
    F.col("unit_price").isNull() |
    (F.trim(F.col("unit_price")) == "") |
    (F.trim(F.col("unit_price")).isin("N/A", "Unknown")) |
    (F.expr("try_cast(unit_price AS DOUBLE)") < 0)
).count()

print("Invalid price count:", invalid_price_count)

Invalid price count: 1


In [148]:
orders.select(
    F.min("order_total").alias("min_order_total"),
    F.max("order_total").alias("max_order_total")
).show()

+---------------+---------------+
|min_order_total|max_order_total|
+---------------+---------------+
|        -45.00 |        invalid|
+---------------+---------------+



In [149]:
numeric_cols = [
    "quantity",
    "unit_price",
    "discount_amount",
    "tax_amount",
    "transaction_amount"
]

df_numeric_profile = transactions.select(
    *[
        F.min(F.col(c)).alias(f"{c}_min")
        for c in numeric_cols
    ],
    *[
        F.max(F.col(c)).alias(f"{c}_max")
        for c in numeric_cols
    ]
)

df_numeric_profile.show()

+------------+--------------+-------------------+--------------+----------------------+------------+--------------+-------------------+--------------+----------------------+
|quantity_min|unit_price_min|discount_amount_min|tax_amount_min|transaction_amount_min|quantity_max|unit_price_max|discount_amount_max|tax_amount_max|transaction_amount_max|
+------------+--------------+-------------------+--------------+----------------------+------------+--------------+-------------------+--------------+----------------------+
|          -2|         10.00|                0.0|          -1.0|                  4.63|         two|           N/A|             224.99|        119.94|             999999.99|
+------------+--------------+-------------------+--------------+----------------------+------------+--------------+-------------------+--------------+----------------------+



In [150]:
from pyspark.sql import functions as F

invalid_quantity_count = transactions.filter(
    F.col("quantity").isNull() |
    (F.trim(F.col("quantity")) == "") |
    F.trim(F.col("quantity")).isin("N/A", "Unknown") |
    (F.expr("try_cast(quantity AS DOUBLE)") <= 0) |
    (
        F.col("quantity").isNotNull() &
        F.expr("try_cast(quantity AS DOUBLE)").isNull()
    )
).count()

print("Invalid quantity count:", invalid_quantity_count)

Invalid quantity count: 3


## Dataset Overview

| Dataset | Rows | Columns | Key | Distinct Keys | Potential Duplicates |
|---|---:|---:|---|---:|---:|
| Customers | 10000 | 9 | `customer_id` | 9999 | 1 |
| Products | 1000 | 9 | `product_id` | 999 | 1 |
| Orders | 30000 | 8 | `order_id` | 29999 | 1 |
| Transactions | 50000 | 10 | `transaction_id` | 49999 | 1 |



## Data Quality Issues

| Dataset | Issue | Column | Count |
|---|---|---|---:|
| Customers | Null/blank IDs | `customer_id` | 0 |
| Products | Invalid price | `unit_price` | 1 |
| Orders | Orphan customer | `customer_id` | 5 |
| Transactions | Invalid quantity | `quantity` | 3 |
| Transactions | Orphan product | `product_id` | 49 |